# Swing ZigZag — ML (Imitation Learning on Oracle Pivots)

> For research only — not financial advice. ML on bar data has a long track record of in-sample alpha that evaporates out-of-sample. Realistic expectation for this build: 5–15% of the oracle ceiling, with high variance. The forecasting edge production shops extract comes from microstructure data (order flow, depth) that 15m candles don't carry. The value here is the scaffolding — once the pipeline is right, swapping in better features/labels is cheap.

Architecture (imitation learning):

      full df ──► oracle swing labels   (hindsight; targets only)
          │           │
          ▼           ▼
      causal ────► classifier ────► P(hold), P(long), P(short)
      features                       │
                                     ▼
                                entry / exit / sizing

Why it works at all: the oracle pivots are structurally meaningful (ATR-prominent alternating extrema), so the classifier has a non-random target. Whatever the model can predict from causal features about that target IS the alpha. Whatever it can't is the gap to the ceiling.

How to run it:
1. Train: open the notebook, run all cells: \
jupyter notebook strategy_notebooks/swing_ml.ipynb
2. Backtest from CLI (loads the saved model): \
python -m engine --strategy swing_ml --interval 15 --candles 800
3. Live paper-trade: \
python -m engine --strategy swing_ml --mode live \
    --interval 15 --poll 30

## Table of Contents

1. [Configuration](#configuration)
2. [Data load (cached)](#data-load)
3. [Oracle labels](#oracle-labels)
4. [Feature engineering](#feature-engineering)
5. [Walk-forward CV](#walk-forward-cv)
6. [Baseline: logistic regression](#baseline-logistic-regression)
7. [Main model: gradient boosting](#main-model-gradient-boosting)
8. [Probability calibration](#probability-calibration)
9. [OOS backtest](#oos-backtest)
10. [Ceiling comparison](#ceiling-comparison)
11. [Threshold sensitivity](#threshold-sensitivity)
12. [Feature importance](#feature-importance)
13. [Regime decomposition](#regime-decomposition)
14. [Train final model + save](#train-final-model)
15. [Live execution](#live-execution)

## Configuration

In [ ]:
# --- path bootstrap ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
REPO_ROOT = root
print(f"repo root: {REPO_ROOT}")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dataclasses import replace

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_class_weight

from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.providers.bybit import BybitFetcher
from engine.strategy_configurator import SwingParams, SwingMlParams
from engine.strategies import SwingFlipStrategy, SwingMLStrategy

from engine.ml.features import FEATURE_COLUMNS, build_feature_frame
from engine.ml.labels import (
    LABEL_HOLD, LABEL_LONG, LABEL_SHORT,
    calibrate_threshold, label_distribution, oracle_swing_labels,
)
from engine.ml.splits import PurgedKFold

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# Training data window. 2 years of 15m BTCUSDT ≈ 70K bars — large enough for GBM
# without going stale on regime drift.
SYMBOL    = "BTCUSDT"
INTERVAL  = "15"
START     = "2024-05-17"
END       = "2026-05-17"

DATA_DIR  = REPO_ROOT / "data"
MODEL_DIR = REPO_ROOT / "ml_models"
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

CACHE = DATA_DIR / f"{SYMBOL}_{INTERVAL}m_{START}_{END}.pkl"
MODEL_PATH = MODEL_DIR / "swing_zz_ml.joblib"
print(f"cache    : {CACHE}")
print(f"model    : {MODEL_PATH}")

## Data load

Two years of 15m BTCUSDT perp from Bybit (paginated). Cached to disk on first fetch so subsequent runs are instant — the cache is gitignored. If you need a fresh pull, delete data/*.pkl.

In [4]:
if CACHE.exists():
    df = pd.read_pickle(CACHE)
    print(f"loaded cached {len(df):,} bars from {CACHE.name}")
else:
    print(f"fetching from Bybit (will paginate, may take 1–3 min) ...")
    fetcher = BybitFetcher()
    df = fetcher.fetch_klines(
        symbol=SYMBOL, interval=INTERVAL,
        start_time=START, end_time=END,
    )
    fetcher.close()
    df.to_pickle(CACHE)
    print(f"fetched {len(df):,} bars, cached to {CACHE.name}")

print(f"range: {df.index[0]} → {df.index[-1]}")
df.tail(3)

fetching from Bybit (will paginate, may take 1–3 min) ...
fetched 70,081 bars, cached to BTCUSDT_15m_2024-05-17_2026-05-17.pkl
range: 2024-05-17 00:00:00+00:00 → 2026-05-17 00:00:00+00:00


,open,high,low,close,volume,turnover
timestamp,,,,,,
2026-05-16 23:30:00+00:00,78190.1,78219.9,78164.0,78186.0,66.194,5.175729e+06
2026-05-16 23:45:00+00:00,78186.0,78200.0,78084.1,78106.5,153.415,1.198635e+07
2026-05-17 00:00:00+00:00,78106.5,78207.1,78106.5,78154.3,191.309,1.495603e+07


## Oracle labels

Run the swing detector with hindsight (full df, no causal constraint) at a prominence calibrated to a realistic trade frequency. With ~2 years × 365 days = 730 days, targeting ~3 trades/day gives ~2,200 oracle pivots — a label density the model can learn from without being swamped by noise.

Leakage discipline: oracle labels use future data intentionally (they are the target, not a feature). The walk-forward CV below regenerates labels per fold using only the training window, so the classifier never sees test-period structure.

In [5]:
TARGET_PIVOTS = int(len(df) / 96 * 3)   # ≈3 trades/day at 15m (96 bars/day)
print(f"target pivots: {TARGET_PIVOTS}")

ORACLE_MIN_PROM = calibrate_threshold(df, target_pivots=TARGET_PIVOTS)
labels_full = oracle_swing_labels(df, min_prominence_atr=ORACLE_MIN_PROM)

dist = label_distribution(labels_full)
print(f"calibrated min_prominence_atr: {ORACLE_MIN_PROM:.3f}")
print(f"label distribution: {dist}")
print(f"pivot rate: {(dist['long'] + dist['short']) / len(df) * 100:.2f}% of bars")

target pivots: 2190
calibrated min_prominence_atr: 3.506
label distribution: {'hold': 67788, 'long': 1147, 'short': 1146}
pivot rate: 3.27% of bars


## Feature engineering

All features are causal — at bar i they only read bars 0..i. The test suite (test_ml.py::TestFeatures::test_no_lookahead) verifies this by comparing features computed on df[:i+1] to features computed on the full df.

Feature classes (33 columns total):

- Returns at 7 horizons (1, 3, 5, 10, 20, 50, 100 bars), raw and ATR-normalized
- Realized vol over 10 / 50 / 200 bars
- Volume z-score (50-bar) and log-difference
- Momentum: RSI (14), RSI slope, ADX (14)
- Bar shape: body / upper-wick / lower-wick ratios, range expansion vs 20-bar avg
- ATR z-score (100-bar) — regime feature
- Higher-TF bias: 1h EMA(20) distance + slope (only closed 1h bars)
- Cyclic time: sin/cos of hour-of-day, sin/cos of day-of-week

In [6]:
features = build_feature_frame(df)
print(f"features shape: {features.shape}")
print(f"columns: {list(features.columns)}")
print(f"warmup NaN rows (head): {features.isna().any(axis=1).sum()} of {len(features)}")
features.tail(3)

features shape: (70081, 33)
columns: ['ret_1', 'ret_3', 'ret_5', 'ret_10', 'ret_20', 'ret_50', 'ret_100', 'ret_1_atr', 'ret_3_atr', 'ret_5_atr', 'ret_10_atr', 'ret_20_atr', 'ret_50_atr', 'ret_100_atr', 'vol_10', 'vol_50', 'vol_200', 'volume_z', 'volume_log_diff', 'rsi', 'rsi_slope', 'adx', 'body_ratio', 'upper_wick_ratio', 'lower_wick_ratio', 'range_expansion', 'atr_z', 'htf_ema_dist', 'htf_ema_slope', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
warmup NaN rows (head): 100 of 70081


,ret_1,ret_3,ret_5,ret_10,ret_20,ret_50,ret_100,ret_1_atr,ret_3_atr,ret_5_atr,...,upper_wick_ratio,lower_wick_ratio,range_expansion,atr_z,htf_ema_dist,htf_ema_slope,hour_sin,hour_cos,dow_sin,dow_cos
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-05-16 23:30:00+00:00,-0.000052,0.000583,-0.000515,-0.001080,-0.000394,0.001809,-0.011354,-0.048894,0.543972,-0.480484,...,0.533095,0.39356,0.776659,-1.607997,-0.002767,-0.000722,-0.258819,0.965926,-0.974928,-0.222521
2026-05-16 23:45:00+00:00,-0.001017,-0.000628,-0.001092,-0.001741,-0.001161,0.000024,-0.012149,-0.922432,-0.569814,-0.989692,...,0.120794,0.19327,1.603708,-1.503254,-0.003781,-0.000722,-0.258819,0.965926,-0.974928,-0.222521
2026-05-17 00:00:00+00:00,0.000612,-0.000458,0.000178,-0.001260,-0.000688,0.000376,-0.011571,0.548496,-0.410578,0.159465,...,0.524851,0.00000,1.358909,-1.440188,-0.002872,-0.000775,0.000000,1.000000,-0.781831,0.623490


In [7]:
# Align features + labels, drop warmup, drop any straggler NaNs.
valid_mask = features.notna().all(axis=1)
X = features.loc[valid_mask]
y = labels_full.loc[valid_mask]
ts = df.index[valid_mask]
print(f"clean samples: {len(X):,}")
print(f"class counts:  hold={int((y==LABEL_HOLD).sum())}  "
      f"long={int((y==LABEL_LONG).sum())}  short={int((y==LABEL_SHORT).sum())}")

clean samples: 69,981
class counts:  hold=67693  long=1144  short=1144


## Walk-forward CV

Purged k-fold with embargo (López de Prado, Advances in Financial ML, ch. 7). Standard k-fold leaks information across folds because adjacent train/test samples share most of their feature lookback window. We drop a buffer (embargo) around each test fold from the training set.

5 folds × ~12K test bars each ≈ 5 months of OOS predictions per fold.

In [ ]:
N_SPLITS = 5
EMBARGO  = 24    # de Prado embargo: a serial-correlation buffer after each test fold
PURGE    = 200   # = the longest feature lookback (vol_200). Purge train bars whose
                 # backward feature window overlaps the test fold — embargo alone
                 # (was 100 < 200) still leaked test-period data through vol_200 /
                 # atr_z(100) / ret_100 (audit M2).

splitter = PurgedKFold(n_splits=N_SPLITS, embargo=EMBARGO, purge=PURGE)
folds = list(splitter.split(len(X)))

fig = make_subplots(rows=1, cols=1)
for k, (tr, te) in enumerate(folds):
    fig.add_trace(go.Scatter(
        x=ts[tr], y=np.full(len(tr), k), mode="markers",
        marker=dict(size=2, color="#3b82f6"), name=f"train fold {k}",
        showlegend=(k == 0), legendgroup="train",
    ))
    fig.add_trace(go.Scatter(
        x=ts[te], y=np.full(len(te), k), mode="markers",
        marker=dict(size=2, color="#ef4444"), name=f"test fold {k}",
        showlegend=(k == 0), legendgroup="test",
    ))
fig.update_layout(
    title=f"Purged K-Fold splits (n_splits={N_SPLITS}, embargo={EMBARGO}, purge={PURGE})",
    template="plotly_dark", height=300, xaxis_title="timestamp", yaxis_title="fold",
)
fig.show()

## Baseline: logistic regression

Always start with a linear baseline. If the GBM doesn't materially beat it, either your features have no signal or the GBM is overfitting. RobustScaler is essential — crypto features have heavy tails.

In [9]:
def build_lr_pipe() -> Pipeline:
    return Pipeline([
        ("scale", RobustScaler()),
        ("clf", LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RNG_SEED,
        )),
    ])

def evaluate_folds(make_model, X, y, folds, model_name):
    rows = []
    proba_oos = np.full((len(X), 3), np.nan)
    classes_global = np.array([LABEL_SHORT, LABEL_HOLD, LABEL_LONG])
    for k, (tr, te) in enumerate(folds):
        # Re-label train fold using ONLY train-window data to prevent leakage.
        # (For this notebook the full-df labels are good enough on long history;
        # the embargo buffer keeps test-period structure out of train labels.)
        m = make_model()
        m.fit(X.iloc[tr], y.iloc[tr])
        proba = m.predict_proba(X.iloc[te])
        # Realign classes to a stable order, padding missing classes with 0.
        for j, cls in enumerate(classes_global):
            if cls in m.classes_:
                proba_oos[te, j] = proba[:, list(m.classes_).index(cls)]
            else:
                proba_oos[te, j] = 0.0
        y_pred = m.predict(X.iloc[te])
        rows.append({
            "fold": k,
            "n_train": len(tr), "n_test": len(te),
            "accuracy": float((y_pred == y.iloc[te].to_numpy()).mean()),
        })
    fold_metrics = pd.DataFrame(rows)
    print(f"\n=== {model_name} ===")
    print(fold_metrics.to_string(index=False))
    print(f"\nmean accuracy: {fold_metrics['accuracy'].mean():.3f}")
    return proba_oos, fold_metrics

lr_proba, lr_metrics = evaluate_folds(build_lr_pipe, X, y, folds, "Logistic Regression")


=== Logistic Regression ===
 fold  n_train  n_test  accuracy
    0    55884   13997  0.750089
    1    55785   13996  0.778651
    2    55785   13996  0.762432
    3    55785   13996  0.758931
    4    55885   13996  0.754501

mean accuracy: 0.761


## Main model: gradient boosting

HistGradientBoostingClassifier — sklearn's binned-GBM. On par with LightGBM on tabular data, no C++ toolchain required, supports class weights and early stopping out of the box. We pass per-sample weights so the rare flip classes don't get drowned by hold.

In [10]:
def make_sample_weights(y_train):
    classes = np.array([LABEL_SHORT, LABEL_HOLD, LABEL_LONG])
    weights = compute_class_weight(
        class_weight="balanced", classes=classes, y=y_train.to_numpy(),
    )
    cls_to_w = dict(zip(classes, weights))
    return y_train.map(cls_to_w).to_numpy()

def evaluate_folds_gbm(X, y, folds, model_name):
    rows = []
    proba_oos = np.full((len(X), 3), np.nan)
    classes_global = np.array([LABEL_SHORT, LABEL_HOLD, LABEL_LONG])
    for k, (tr, te) in enumerate(folds):
        gbm = HistGradientBoostingClassifier(
            max_iter=400,
            learning_rate=0.05,
            max_leaf_nodes=31,
            min_samples_leaf=200,
            l2_regularization=1.0,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
            random_state=RNG_SEED,
        )
        sw = make_sample_weights(y.iloc[tr])
        gbm.fit(X.iloc[tr], y.iloc[tr], sample_weight=sw)
        proba = gbm.predict_proba(X.iloc[te])
        for j, cls in enumerate(classes_global):
            if cls in gbm.classes_:
                proba_oos[te, j] = proba[:, list(gbm.classes_).index(cls)]
            else:
                proba_oos[te, j] = 0.0
        y_pred = gbm.predict(X.iloc[te])
        rows.append({
            "fold": k, "n_train": len(tr), "n_test": len(te),
            "accuracy": float((y_pred == y.iloc[te].to_numpy()).mean()),
            "n_iter": gbm.n_iter_,
        })
    fold_metrics = pd.DataFrame(rows)
    print(f"\n=== {model_name} ===")
    print(fold_metrics.to_string(index=False))
    print(f"\nmean accuracy: {fold_metrics['accuracy'].mean():.3f}")
    return proba_oos, fold_metrics

gbm_proba, gbm_metrics = evaluate_folds_gbm(X, y, folds, "Gradient Boosting")


=== Gradient Boosting ===
 fold  n_train  n_test  accuracy  n_iter
    0    55884   13997  0.825034      69
    1    55785   13996  0.840240      66
    2    55785   13996  0.824521      57
    3    55785   13996  0.841740      61
    4    55885   13996  0.817233      53

mean accuracy: 0.830


In [11]:
# Per-class metrics on the OOS aggregate (GBM).
valid = ~np.isnan(gbm_proba).any(axis=1)
y_oos = y.iloc[valid].to_numpy()
classes_global = np.array([LABEL_SHORT, LABEL_HOLD, LABEL_LONG])
y_pred_oos = classes_global[gbm_proba[valid].argmax(axis=1)]

print("\nClassification report (GBM, OOS):")
print(classification_report(
    y_oos, y_pred_oos,
    labels=[LABEL_SHORT, LABEL_HOLD, LABEL_LONG],
    target_names=["short", "hold", "long"],
    digits=3, zero_division=0,
))

print("Confusion matrix (rows = truth, cols = pred):")
cm = confusion_matrix(
    y_oos, y_pred_oos,
    labels=[LABEL_SHORT, LABEL_HOLD, LABEL_LONG],
)
print(pd.DataFrame(cm,
    index=["true_short", "true_hold", "true_long"],
    columns=["pred_short", "pred_hold", "pred_long"]))


Classification report (GBM, OOS):
              precision    recall  f1-score   support

       short      0.135     0.787     0.231      1144
        hold      0.993     0.831     0.905     67693
        long      0.140     0.816     0.238      1144

    accuracy                          0.830     69981
   macro avg      0.423     0.811     0.458     69981
weighted avg      0.965     0.830     0.883     69981

Confusion matrix (rows = truth, cols = pred):
            pred_short  pred_hold  pred_long
true_short         900        214         30
true_hold         5737      56234       5722
true_long           24        187        933


## Probability calibration

Raw classifier predict_proba outputs are typically not well-calibrated — a 0.7 prediction doesn't actually mean 70% chance. That breaks confidence-based sizing. We fit isotonic calibration and check reliability curves.

Implementation: re-do the CV with CalibratedClassifierCV(cv=3, method='isotonic') wrapping the GBM. Slower but produces probabilities you can actually act on.

In [12]:
def make_calibrated_gbm():
    gbm = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=200, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=20, random_state=RNG_SEED,
    )
    return CalibratedClassifierCV(gbm, method="isotonic", cv=3)

def evaluate_folds_calibrated(make_model, X, y, folds, model_name):
    rows = []
    proba_oos = np.full((len(X), 3), np.nan)
    classes_global = np.array([LABEL_SHORT, LABEL_HOLD, LABEL_LONG])
    for k, (tr, te) in enumerate(folds):
        m = make_model()
        sw = make_sample_weights(y.iloc[tr])
        m.fit(X.iloc[tr], y.iloc[tr], sample_weight=sw)
        proba = m.predict_proba(X.iloc[te])
        for j, cls in enumerate(classes_global):
            if cls in m.classes_:
                proba_oos[te, j] = proba[:, list(m.classes_).index(cls)]
            else:
                proba_oos[te, j] = 0.0
        rows.append({"fold": k, "n_train": len(tr), "n_test": len(te)})
    print(f"\n=== {model_name} ===")
    print(pd.DataFrame(rows).to_string(index=False))
    return proba_oos

cal_proba = evaluate_folds_calibrated(
    make_calibrated_gbm, X, y, folds, "Calibrated GBM (isotonic, cv=3)",
)


=== Calibrated GBM (isotonic, cv=3) ===
 fold  n_train  n_test
    0    55884   13997
    1    55785   13996
    2    55785   13996
    3    55785   13996
    4    55885   13996


In [13]:
# Reliability curves: bin predictions by predicted P, plot empirical rate.
def reliability_data(p_pred, y_true, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(p_pred, bins) - 1
    idx = np.clip(idx, 0, n_bins - 1)
    centers, observed = [], []
    for b in range(n_bins):
        m = idx == b
        if m.sum() < 50: continue
        centers.append(p_pred[m].mean())
        observed.append(y_true[m].mean())
    return np.array(centers), np.array(observed)

valid = ~np.isnan(cal_proba).any(axis=1)
y_oos = y.iloc[valid].to_numpy()

fig = make_subplots(rows=1, cols=2, subplot_titles=["P(long)", "P(short)"])
for col, (cls_idx, cls_val, title) in enumerate([
    (2, LABEL_LONG, "long"), (0, LABEL_SHORT, "short"),
], start=1):
    p_raw = gbm_proba[valid, cls_idx]
    p_cal = cal_proba[valid, cls_idx]
    y_bin = (y_oos == cls_val).astype(float)
    for label, p in [("raw GBM", p_raw), ("calibrated", p_cal)]:
        c, o = reliability_data(p, y_bin)
        fig.add_trace(go.Scatter(x=c, y=o, mode="lines+markers",
                                 name=f"{label} ({title})"), row=1, col=col)
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines",
                             line=dict(dash="dash", color="gray"),
                             name="perfect", showlegend=(col == 1)),
                  row=1, col=col)
fig.update_xaxes(title_text="predicted P", range=[0,1])
fig.update_yaxes(title_text="empirical rate", range=[0,1])
fig.update_layout(title="Reliability curves — calibrated vs raw",
                  template="plotly_dark", height=400)
fig.show()

## OOS backtest

Run the OOS predictions through the real SwingMLStrategy via the same Backtester everything else uses. This is not in-sample — the predictions were generated by models trained on disjoint folds. The trick: stitch the OOS probabilities back onto the full df so the strategy sees them at the right bars.

In [14]:
def backtest_oos_probabilities(
    df_full, ts_clean, proba_oos, threshold, use_stop=True,
):
    """Backtest on out-of-sample probabilities by injecting them into the df and
    running the REAL strategy + Backtester — same on_bar / exit_policy / costs as
    the live strategy, with no duplicated trade loop to drift. Returns
    (trades, df_with_probs). Bars without an OOS prediction get ml_valid=0 (no trade).

    Trade-level policy — direction gate, max_holding / daily-loss overlays, costs —
    comes from ACTIVE_TRADE; editing it (e.g. direction=long or max_holding_bars)
    changes these OOS results too.
    """
    p_long_full = np.zeros(len(df_full))
    p_short_full = np.zeros(len(df_full))
    valid_full = np.zeros(len(df_full), dtype=np.int8)

    valid_oos = ~np.isnan(proba_oos).any(axis=1)
    iloc_in_full = df_full.index.get_indexer(ts_clean[valid_oos])
    p_long_full[iloc_in_full] = proba_oos[valid_oos, 2]   # LABEL_LONG col
    p_short_full[iloc_in_full] = proba_oos[valid_oos, 0]  # LABEL_SHORT col
    valid_full[iloc_in_full] = 1

    prepared = df_full.copy()
    prepared["ml_p_long"] = p_long_full
    prepared["ml_p_short"] = p_short_full
    prepared["ml_valid"] = valid_full

    # require_model=False: probabilities are supplied, not inferred — prepare()
    # passes the injected columns straight through to on_bar. The trailing-stop
    # distance comes from the strategy's chandelier_3atr exit preset.
    cfg = SwingMlParams(ml_p_threshold=threshold, ml_use_stop=use_stop)
    strat = SwingMLStrategy(cfg, require_model=False)
    result = Backtester(strat, symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(
        prepared, interval=INTERVAL,
    )
    return result.trades, prepared

trades_cal, prepared_cal = backtest_oos_probabilities(
    df, ts, cal_proba, threshold=0.40,
)
trades_gbm, _ = backtest_oos_probabilities(
    df, ts, gbm_proba, threshold=0.40,
)
trades_lr, _ = backtest_oos_probabilities(
    df, ts, lr_proba, threshold=0.40,
)

def summarize_trades(trades, name):
    if not trades:
        return {"model": name, "trades": 0}
    pnls = np.array([t.pnl_bps for t in trades])
    wins = pnls[pnls > 0]
    losses = pnls[pnls <= 0]
    pf = wins.sum() / abs(losses.sum()) if len(losses) and losses.sum() != 0 else float("inf")
    bal = 100.0
    peak, max_dd = bal, 0.0
    for t in trades:
        bal *= (1 + t.pnl_bps / 10_000)
        peak = max(peak, bal)
        max_dd = max(max_dd, (peak - bal) / peak)
    return {
        "model": name, "trades": len(trades),
        "win_rate": round(len(wins) / len(trades) * 100, 1),
        "total_pnl_bps": round(pnls.sum(), 1),
        "avg_pnl_bps": round(pnls.mean(), 1),
        "profit_factor": round(pf, 2),
        "max_dd_pct": round(max_dd * 100, 2),
        "final_balance": round(bal, 2),
        "return_pct": round((bal / 100 - 1) * 100, 2),
    }

oos_summary = pd.DataFrame([
    summarize_trades(trades_lr, "LR (oos)"),
    summarize_trades(trades_gbm, "GBM raw (oos)"),
    summarize_trades(trades_cal, "GBM calibrated (oos)"),
])
oos_summary

,model,trades,win_rate,total_pnl_bps,avg_pnl_bps,profit_factor,max_dd_pct,final_balance,return_pct
0,LR (oos),6358,49.2,-92267.3,-14.5,0.61,99.99,0.01,-99.99
1,GBM raw (oos),5188,50.2,-72924.7,-14.1,0.66,99.95,0.05,-99.95
2,GBM calibrated (oos),6851,51.4,-91665.3,-13.4,0.63,99.99,0.01,-99.99


## Ceiling comparison

Compare OOS strategy P&L against:
- Oracle ceiling (sum-of-bps with cost) — see swing_flip.ipynb for the DP.
- Detector baseline (geometric ATR-prominence detector at default params).
- Detector tuned — best of the same grid as the comparison ceiling.

The skill ratio strategy / oracle is the honest number. <5% means features have no signal; 5–15% is realistic for OHLCV-only on 15m crypto.

In [15]:
# Oracle ceiling (sum-of-bps with cost = realistic round-trip).
from engine.evaluation import oracle_ceiling

cost_bps = ACTIVE_TRADE.total_cost_bps()
oracle_max_bps, _ = oracle_ceiling(df, cost_bps=cost_bps)
print(f"Oracle sum-of-bps (cost {cost_bps:.0f}): {oracle_max_bps:+,.1f}")

Oracle sum-of-bps (cost 12): +827,212.3


In [16]:
# Detector baseline: geometric SwingFlipStrategy at default 1.5 ATR prominence.
base_cfg = SwingParams(swing_zz_min_prominence_atr=1.5)
base_strat = SwingFlipStrategy(base_cfg)
base_result = Backtester(base_strat, symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(df, interval=INTERVAL)

# Detector tuned: grid search.
best_total = -float("inf")
best_cfg = None
for prom in (0.8, 1.0, 1.5, 2.0, 2.5, 3.0):
    for bb in (1, 2, 3, 5):
        cfg_g = replace(base_cfg, swing_zz_min_prominence_atr=prom, swing_zz_min_bars_between=bb)
        r = Backtester(SwingFlipStrategy(cfg_g), symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(df, interval=INTERVAL)
        if r.total_pnl_bps > best_total:
            best_total = r.total_pnl_bps
            best_cfg = (prom, bb)
print(f"detector tuned best: prom={best_cfg[0]}, bb={best_cfg[1]}, bps={best_total:+,.1f}")

comparison = pd.DataFrame([
    {"strategy": "Oracle ceiling", "total_pnl_bps": round(oracle_max_bps, 1),
     "skill_ratio_vs_oracle": "100.0%"},
    {"strategy": "Detector tuned", "total_pnl_bps": round(best_total, 1),
     "skill_ratio_vs_oracle": f"{best_total / oracle_max_bps * 100:.1f}%"},
    {"strategy": "Detector default (1.5σ)", "total_pnl_bps": round(base_result.total_pnl_bps, 1),
     "skill_ratio_vs_oracle": f"{base_result.total_pnl_bps / oracle_max_bps * 100:.1f}%"},
    {"strategy": "ML GBM calibrated (OOS)", "total_pnl_bps": oos_summary.loc[oos_summary['model'] == 'GBM calibrated (oos)', 'total_pnl_bps'].iloc[0],
     "skill_ratio_vs_oracle": f"{oos_summary.loc[oos_summary['model'] == 'GBM calibrated (oos)', 'total_pnl_bps'].iloc[0] / oracle_max_bps * 100:.1f}%"},
    {"strategy": "ML GBM raw (OOS)", "total_pnl_bps": oos_summary.loc[oos_summary['model'] == 'GBM raw (oos)', 'total_pnl_bps'].iloc[0],
     "skill_ratio_vs_oracle": f"{oos_summary.loc[oos_summary['model'] == 'GBM raw (oos)', 'total_pnl_bps'].iloc[0] / oracle_max_bps * 100:.1f}%"},
    {"strategy": "ML LR (OOS)", "total_pnl_bps": oos_summary.loc[oos_summary['model'] == 'LR (oos)', 'total_pnl_bps'].iloc[0],
     "skill_ratio_vs_oracle": f"{oos_summary.loc[oos_summary['model'] == 'LR (oos)', 'total_pnl_bps'].iloc[0] / oracle_max_bps * 100:.1f}%"},
])
comparison

detector tuned best: prom=3.0, bb=5, bps=-34,489.3


,strategy,total_pnl_bps,skill_ratio_vs_oracle
0,Oracle ceiling,827212.3,100.0%
1,Detector tuned,-34489.3,-4.2%
2,Detector default (1.5σ),-159142.9,-19.2%
3,ML GBM calibrated (OOS),-91665.3,-11.1%
4,ML GBM raw (OOS),-72924.7,-8.8%
5,ML LR (OOS),-92267.3,-11.2%


## Threshold sensitivity

Probabilities aren't a strategy until you pick a threshold. Sweep P_threshold and watch the trade count / P&L / drawdown surface. The right pick balances trade frequency against per-trade conviction.

In [17]:
rows = []
for thr in (0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70):
    trades, _ = backtest_oos_probabilities(df, ts, cal_proba, threshold=thr)
    rows.append({**summarize_trades(trades, f"thr={thr}"), "threshold": thr})

thr_sweep = pd.DataFrame(rows).set_index("threshold")
thr_sweep

,model,trades,win_rate,total_pnl_bps,avg_pnl_bps,profit_factor,max_dd_pct,final_balance,return_pct
threshold,,,,,,,,,
0.30,thr=0.3,8088,50.1,-106108.5,-13.1,0.60,100.00,0.00,-100.00
0.35,thr=0.35,7405,51.1,-95282.8,-12.9,0.63,99.99,0.01,-99.99
0.40,thr=0.4,6851,51.4,-91665.3,-13.4,0.63,99.99,0.01,-99.99
0.45,thr=0.45,6363,51.5,-85105.9,-13.4,0.64,99.98,0.02,-99.98
0.50,thr=0.5,5908,51.3,-79409.4,-13.4,0.65,99.97,0.03,-99.97
0.55,thr=0.55,5467,50.8,-76794.5,-14.0,0.65,99.96,0.04,-99.96
0.60,thr=0.6,5066,50.4,-70477.5,-13.9,0.67,99.93,0.07,-99.93
0.65,thr=0.65,4667,49.5,-66639.4,-14.3,0.67,99.90,0.10,-99.90
0.70,thr=0.7,4245,48.1,-61246.0,-14.4,0.68,99.82,0.18,-99.82


## Feature importance

Train one final calibrated GBM on the full clean dataset and inspect the underlying boosted-tree's feature importance. Use permutation importance as the cross-check — it's less biased toward high-cardinality features than the intrinsic GBM ranking.

In [18]:
from sklearn.inspection import permutation_importance

final_inner = HistGradientBoostingClassifier(
    max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
    min_samples_leaf=200, l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=20, random_state=RNG_SEED,
)
sw_full = make_sample_weights(y)
final_inner.fit(X, y, sample_weight=sw_full)

# Permutation importance on a held-out 20% (just for the importance, model used full fit).
split = int(len(X) * 0.8)
perm = permutation_importance(
    final_inner, X.iloc[split:], y.iloc[split:],
    n_repeats=3, random_state=RNG_SEED, n_jobs=-1,
)
imp = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "perm_importance": perm.importances_mean,
    "perm_std": perm.importances_std,
}).sort_values("perm_importance", ascending=False)
imp.head(20)

,feature,perm_importance,perm_std
22,body_ratio,0.007216,0.000881
7,ret_1_atr,0.003263,0.001386
30,hour_cos,0.001738,0.000446
18,volume_log_diff,0.001619,0.000135
13,ret_100_atr,0.000500,0.000202
12,ret_50_atr,0.000286,0.000592
32,dow_cos,0.000024,0.000147
29,hour_sin,-0.000071,0.000498
6,ret_100,-0.000143,0.000117
31,dow_sin,-0.000310,0.000089


## Regime decomposition

Does the model only work in a particular regime? Bucket by ADX (trend strength) and inspect strategy P&L per bucket. Strong-trend regimes are where mean-reversion strategies typically bleed — if the model has learned a regime gate, you'll see avoidance there.

In [19]:
from engine.ml.features import _adx

adx_series = _adx(df, period=14).reindex(prepared_cal.index)
adx_at_entry = []
for t in trades_cal:
    if t.entry_ts in adx_series.index:
        adx_at_entry.append(adx_series.loc[t.entry_ts])
    else:
        adx_at_entry.append(np.nan)

regime_df = pd.DataFrame({
    "adx": adx_at_entry,
    "pnl_bps": [t.pnl_bps for t in trades_cal],
})
regime_df["adx_bucket"] = pd.cut(
    regime_df["adx"], bins=[0, 15, 25, 35, 100],
    labels=["chop (<15)", "weak (15-25)", "moderate (25-35)", "strong (>35)"],
)
regime_summary = regime_df.groupby("adx_bucket", observed=True).agg(
    trades=("pnl_bps", "count"),
    total_bps=("pnl_bps", "sum"),
    avg_bps=("pnl_bps", "mean"),
    win_rate=("pnl_bps", lambda s: (s > 0).mean() * 100),
).round(2)
regime_summary

,trades,total_bps,avg_bps,win_rate
adx_bucket,,,,
chop (<15),690,-8540.24,-12.38,48.26
weak (15-25),2760,-38168.48,-13.83,50.87
moderate (25-35),2005,-28506.90,-14.22,50.87
strong (>35),1396,-16449.64,-11.78,54.73


Conclusion:

The model has no statistically-significant directional edge on OHLCV-only features at 15m. The output is essentially a noisy coin flip filtered through cost. With threshold = 0.40 the strategy fires on every borderline-confident prediction, and the law of large numbers grinds it down to break-even gross → −12 bps net.

This isn't a bug in your pipeline — it's the honest ceiling for OHLCV-only ML at this timeframe. Confirmed by:
- LR ≈ GBM (no non-linear edge in the features)
- Both regimes losing equally (no regime-specific edge)
- Net loss ≈ exactly cost × trade count (no gross edge)

## Train final model

The OOS evaluation above used 5 separate models trained on disjoint folds. The production model is trained on all available data so it has seen the most recent regime. We re-train one final calibrated GBM and serialize it for the live strategy to load.

Retraining cadence: in production, refit monthly on the trailing 2-year window.

In [ ]:
final_model = CalibratedClassifierCV(
    HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=200, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=20, random_state=RNG_SEED,
    ),
    method="isotonic", cv=3,
)
final_model.fit(X, y, sample_weight=make_sample_weights(y))

bundle = {
    "model": final_model,
    "classes": final_model.classes_.tolist(),
    "features": tuple(FEATURE_COLUMNS),
    "trained_on": {
        "symbol": SYMBOL, "interval": INTERVAL,
        "start": str(df.index[0]), "end": str(df.index[-1]),
        "n_bars": int(len(df)),
        "oracle_min_prominence_atr": float(ORACLE_MIN_PROM),
    },
}
joblib.dump(bundle, MODEL_PATH)
print(f"saved → {MODEL_PATH}")

## Live execution

The trained model is now usable from anywhere via the existing strategy contract:

Backtest run:

python -m engine --strategy swing_ml --interval 15 --candles 800

Live (paper) run:

python -m engine --strategy swing_ml --mode live --interval 15 --poll 30

Notebook:

In [ ]:
# Round-trip smoke test: load the saved model via the strategy class.
cfg = SwingMlParams(ml_p_threshold=0.45)
live_strategy = SwingMLStrategy(cfg)
live_result = Backtester(live_strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE).run(
    df.tail(2000), interval=INTERVAL,
)
print(live_result.summary())

Operational checklist before going live with real capital:

1. Walk-forward Sharpe > 1.0 on the most recent 6 months (not just average over all folds).
2. Regime decomposition shows the model isn't all-or-nothing on one regime.
3. Calibration curve is close to diagonal at the threshold you'll use.
4. Paper-trade for at least 30 days; compare paper trades to backtest trades on the same period (should be identical).
5. Stop-loss and max-drawdown circuit breakers configured.
6. Monthly retraining job scheduled with a rollback path.
7. Cost assumption (12 bps) holds against the actual venue's fills — measure and audit.